<a href="https://colab.research.google.com/github/riyu658/olist-ecommerce-analytics/blob/main/04_insights_conclusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

In [ ]:
master = pd.read_csv('/content/drive/My Drive/olist/olist_master.csv', parse_dates=['order_purchase_timestamp'])
df = master[(master['order_status'] == 'delivered') &
            (master['delivery_days'].notna()) &
            (master['delivery_days'] >= 0) &
            (master['delivery_days'] <= 60)].copy()

print(f"Analysis frame: {len(df):,} orders")

Analysis frame: 96,182 orders


##**Executive Summary**

*   Analysis of ~96,000 delivered orders on the Olist marketplac (2016–2018) reveals that **delivery reliability, not price or freight cost, is the dominant driver of customer satisfaction**.
*   Late deliveries cut average review scores nearly in half (4.29 → 2.28), while freight cost has almost no effect (4.21 → 4.04). The business should therefore prioritize logistics improvements over shipping subsidies.

## 🔑 **Key Insights**

###**1. Delivery speed is the #1 satisfaction driver**
Late orders average **2.28** vs **4.29** for on-time orders. **52.3% of late
deliveries receive a 1-star review** (vs 6.6% on-time). Delivery performance is
the single strongest lever Olist has over its reputation.


###**2. Reviews are polarized**
Scores cluster at 1★ and 5★ — customers rarely feel neutral. This means service
failures create vocal detractors, while good experiences create promoters.
Little middle ground.

###**3. Product category matters for satisfaction**
Lowest-rated categories: **computer_accessories, furniture_decor, telephony**
(≥200 orders). Likely shipping-sensitivity (fragile goods) or quality issues —
candidates for seller QA programs.

###**4. Volume concentrates in home & personal care**
**bed_bath_table** and **health_beauty** dominate order volume — the marketplace's
core revenue engine.

###**5. Extreme geographic concentration**
**São Paulo (SP) alone accounts for ~42% of orders.** Growth opportunity lies in
under-penetrated regions (North/Northeast), though logistics costs there are higher.

###**6. Credit card is king**
Credit card dominates payments, with **~2.9 installments on average** — Brazilian
consumers strongly favor installment plans. BNPL/boleto options remain secondary.

###**7. Freight is structurally unfair to cheap & remote orders**
Freight is flat (R$20–80) regardless of item price, so low-value items and
remote northern states (RR, AP, AM) pay the most *relative* to product value.

###**8. Freight cost barely hurts satisfaction — speed does**
Across freight buckets, scores fall only 4.21 → 4.04. Lateness (−2 pts) hurts
~10× more than freight cost (−0.2 pts). **Fix delays, not freight prices.**

###**9. Retention is alarmingly low — only 3.0% of customers ever place a second order**
The marketplace wins transactions but not loyalty. Combined with polarized
reviews, every service failure permanently loses a customer.

In [ ]:
# Verify all headline numbers in one place
print("=== INSIGHT 1: Delivery vs reviews ===")
print(f"On-time avg score: {df.loc[~df['was_late'],'review_score'].mean():.2f}")
print(f"Late avg score:    {df.loc[df['was_late'],'review_score'].mean():.2f}")
print(f"1-star rate on-time: {(df.loc[~df['was_late'],'review_score']==1).mean()*100:.1f}%")
print(f"1-star rate late:    {(df.loc[df['was_late'],'review_score']==1).mean()*100:.1f}%")

print("\n=== INSIGHT 5: Geography ===")
top_state = df['customer_state'].value_counts()
print(f"{top_state.index[0]}: {top_state.iloc[0]:,} orders ({top_state.iloc[0]/len(df)*100:.0f}%)")

print("\n=== INSIGHT 6: Payments ===")
print(f"Avg installments: {df['payment_installments'].mean():.1f}")

print("\n=== INSIGHT 8: Freight buckets ===")
df['freight_pct'] = df['total_freight'] / df['total_price'] * 100
df['freight_bucket'] = pd.cut(df['freight_pct'], bins=[0, 10, 25, 50, 100, np.inf],
                              labels=['<10%', '10-25%', '25-50%', '50-100%', '>100%'])
print(df.groupby('freight_bucket', observed=True)['review_score'].agg(['mean', 'count']))

=== INSIGHT 1: Delivery vs reviews ===
On-time avg score: 4.29
Late avg score:    2.28
1-star rate on-time: 6.6%
1-star rate late:    52.3%

=== INSIGHT 5: Geography ===
SP: 40,441 orders (42%)

=== INSIGHT 6: Payments ===
Avg installments: 2.9

=== INSIGHT 8: Freight buckets ===
                    mean  count
freight_bucket                 
<10%            4.206784  14769
10-25%          4.175966  37797
25-50%          4.143512  27503
50-100%         4.126167  12103
>100%           4.041393   3044


## 🎯 **Business Recommendations**

1. **Prioritize logistics over price subsidies.** Redirect budget from freight
   discounts toward carrier SLAs, especially on long-distance routes — a 1-point
   review gain is worth more than a R$5 freight cut.

2. **Tighten delivery estimates.** 6.6% of orders arrive late, but the *estimated*
   date is conservative (median delivery 12 days early). Tighter, honest estimates
   would convert "pleasantly early" into operational efficiency.

3. **Flag fragile-category sellers.** Computer accessories, furniture and telephony
   underperform — introduce packaging standards or seller scorecards for these categories.

4. **Grow beyond São Paulo.** With ~42% of orders in one state, expansion into the
   North/Northeast (via regional fulfillment) is the biggest untapped growth lever.

5. **Lean into installments.** Push credit-card installment options at checkout —
   it's the payment behavior Brazilian customers already prefer.

6. **Protect low-value orders.** Flat freight makes sub-R$50 orders uneconomical
   (freight can exceed item price). Consider minimum-order incentives or
   consolidated shipping for these.

## ⚠️ **Limitations**

- **Snapshot bias:** Data covers 2016–2018 only; marketplace conditions have since changed.
- **Missing context:** No cost/profit data — we see revenue and satisfaction, not margins.
- **Review non-response bias:** Only ~40% of orders have reviews; unhappy customers
  may be over-represented.
- **Translation gaps:** A few Portuguese categories mapped to 'unknown'.
- **Freight insight caveat:** The &gt;100% freight bucket holds only ~3k orders (3%) —
  effect is small and directionally, not statistically, established.

## 🏁 **Conclusion**

On a marketplace, **experience beats economics**. Customers punish lateness
(~2-point score drop) but shrug at freight prices (~0.2-point drop). Olist's
fastest path to higher review scores — and the revenue that follows — runs
through its logistics network, not its pricing.